# ***Proyecto final - Data Science I***

*Predicción de ventas de una cafeteria*

En este proyecto se analiza un dataset de ventas de una cafetería y se construye un modelo de machine learning para predecir el total de venta a partir de distintas variables.


In [75]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [76]:
df = pd.read_csv("/content/drive/MyDrive/Repositorio/Entrega 1/Entrega1-cafe.csv", encoding='latin1', low_memory=False, sep=';')

El dataset contiene información sobre las ventas realizadas en una cafetería.
Cada fila representa una venta y las columnas incluyen datos como el producto, el precio, la cantidad y el total de la venta.
La variable objetivo del proyecto será el total de venta, por lo que se trata de un problema de regresión.

In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID venta            49 non-null     float64
 1   Fecha               49 non-null     object 
 2   Dia_semana          49 non-null     object 
 3   Hora                49 non-null     object 
 4   Producto            49 non-null     object 
 5   Categoria producto  49 non-null     object 
 6   Precio              50 non-null     object 
 7   Descuento           49 non-null     object 
 8   Metodo de pago      49 non-null     object 
 9   Total venta         49 non-null     object 
 10  Genero cliente      49 non-null     object 
 11  Edad cliente        49 non-null     float64
 12  Cantidad            49 non-null     float64
 13  ID cliente          49 non-null     object 
 14  Ciudad              49 non-null     object 
dtypes: float64(3), object(12)
memory usage: 120.0+ MB

In [78]:
df.describe()

,ID venta,Edad cliente,Cantidad
count,49.000000,49.000000,49.000000
mean,3.714286,28.489796,1.979592
std,2.549510,7.027098,1.089530
min,1.000000,20.000000,1.000000
25%,2.000000,22.000000,1.000000
50%,3.000000,30.000000,2.000000
75%,6.000000,34.000000,2.000000
max,9.000000,40.000000,4.000000


In [79]:
df = df.drop(columns=["Fecha"])

In [80]:
# Clean 'Total venta' by removing '$' and '.' before converting to numeric
df['Total venta'] = df['Total venta'].astype(str).str.replace('$', '', regex=False).str.replace('.', '', regex=False)
# Convert 'Total venta' to numeric, coercing errors to NaN
df['Total venta'] = pd.to_numeric(df['Total venta'], errors='coerce')

# Impute missing values in 'Total venta' with the median
median_total_venta = df['Total venta'].median()
df['Total venta'].fillna(median_total_venta, inplace=True)

df = pd.get_dummies(df, drop_first=True)

/tmp/ipython-input-750476697.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Total venta'].fillna(median_total_venta, inplace=True)


In [81]:
X = df.drop("Total venta", axis=1)
y = df["Total venta"]

In [82]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [83]:
print("Unique values in 'Total venta':\n", df['Total venta'].unique())
print("Data type of 'Total venta':", df['Total venta'].dtype)
print("Value counts for 'Total venta':\n", df['Total venta'].value_counts(dropna=False))

Unique values in 'Total venta':
 [3000. 3500. 4200. 5000. 3200.]
Data type of 'Total venta': float64
Value counts for 'Total venta':
 Total venta
3500.0    1048535
3000.0         15
4200.0         11
5000.0         10
3200.0          4
Name: count, dtype: int64


En esta sección se aplica una técnica de selección de características para reducir la cantidad de variables utilizadas por el modelo.

In [85]:
# Impute missing values in numerical columns of X_train and X_test
# Use median for imputation as it's robust to outliers
for col in X_train.select_dtypes(include=['number']).columns:
    median_val = X_train[col].median()
    X_train[col].fillna(median_val, inplace=True)
    X_test[col].fillna(median_val, inplace=True)

selector = SelectKBest(score_func=f_regression, k=5)

X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

/tmp/ipython-input-569189005.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train[col].fillna(median_val, inplace=True)
/tmp/ipython-input-569189005.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using '

In [86]:
selected_features = X.columns[selector.get_support()]
selected_features


Index(['ID venta', 'Producto_Bombon', 'Producto_Irlandes', 'Precio_$ 4.200',
       'Precio_$ 5.000'],
      dtype='object')

Se entrena un modelo de regresión lineal utilizando las variables seleccionadas.


In [87]:
model = LinearRegression()
model.fit(X_train_selected, y_train)


LinearRegression()

In [88]:
y_pred = model.predict(X_test_selected)


In [89]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

mae, mse, rmse, r2


(0.018391983680849038,
 4.738158456170008,
 np.float64(2.1767311400744944),
 0.8321511909504553)

El MAE representa el error promedio de las predicciones.
El RMSE penaliza los errores grandes.
El valor de R² indica qué proporción de la variabilidad del total de ventas es explicada por el modelo.
Los resultados obtenidos muestran un desempeño aceptable para este tipo de problema.


En este proyecto se construyó un modelo de regresión para predecir el total de venta de una cafetería.
La selección de características permitió reducir la complejidad del modelo.
Las métricas obtenidas indican que el modelo logra capturar relaciones relevantes en los datos.
Como mejora futura, se podrían probar modelos más complejos o incorporar nuevas variables.
